# 04 — Multiclass Decoding (all four conditions)

**Purpose:** Move from two-class to **four-class** decoding. The decoder must now
assign each trial to one of `pos_val`, `neg_val`, `pos_aro`, `neg_aro`. This is
harder than any binary problem, chance drops to **25%**, and the most useful
output is no longer a single accuracy but a **confusion matrix** that shows
*which* conditions get mixed up.

> Reference: Nilearn's decoding tutorials include multiclass examples; the
> `Decoder` object handles multiclass automatically (one-vs-rest under the hood),
> so most of your binary pipeline carries straight over.

---

## Notebook overview

1. Load `decoder_4class.csv` (all 64 trials, four labels).
2. Build `imgs`, `y`, `groups`.
3. Reuse the decoder pipeline — it goes multiclass with no special handling.
4. Score against chance = 25%.
5. Build and read a **confusion matrix** to interpret the decoder's mistakes.

## What you are learning

- What **multiclass classification** is and why it is harder than binary.
- Why chance = **1 / n_classes** = 25% here.
- How to build, plot, and *read* a **confusion matrix**.
- How to turn "the decoder's mistakes" into a scientific observation.

## Why this matters scientifically

Overall 4-class accuracy is a blunt summary. The *pattern of confusions* is the
interesting part: if the decoder reliably tells valence-conditions from
arousal-conditions but muddles positive vs negative *within* a dimension, that
tells you the dimension is more strongly represented than the sign. A confusion
matrix turns a single number into a structured hypothesis about representational
geometry.

---
## Section 1 — From Binary to Multiclass

A binary SVM draws one boundary. With four classes, scikit-learn (and Nilearn's
`Decoder`) typically trains **one-vs-rest**: a separate "is it this class or
not?" SVM per class, then predicts whichever is most confident. You do not have
to code this — passing four distinct labels in `y` is enough.

Why is it harder?

- More ways to be wrong: each trial can be misassigned to any of three other
  classes, not just one.
- The data is spread thinner: 16 trials per class, and within each CV fold the
  training set is smaller still.
- **Chance = 1 / number_of_classes = 1 / 4 = 25%.** A decoder scoring 30% is
  doing *something*, even though 30% sounds low — context (the 25% bar) is
  everything.

---
## Section 2 — Load the 4-Class Table and Build Inputs

**TODO:**
- [ ] Read `decoder_4class.csv` into `table`.
- [ ] Confirm `table["label"].value_counts()` shows 16 for each of the four classes.
- [ ] Build `imgs`, `y`, `groups`; assert equal lengths.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

subject = "sub-097"
project_dir  = Path(r"C:\ManzaRotation")
decoding_dir = project_dir / "Analysis" / "outputs" / subject / "decoding"
tables_dir   = decoding_dir / "tables"

# TODO: read decoder_4class.csv into `table`
# TODO: check 16 per class
# TODO: build imgs, y, groups

---
## Section 3 — Reuse the Pipeline (Now Multiclass)

The same `Decoder` configuration works. Two small notes:

- `scoring="accuracy"` still makes sense, but with four classes you may also want
  per-class scores — `decoder.cv_scores_` is keyed by class, so you get those for
  free.
- Keep `LeaveOneGroupOut` so each fold still tests on a held-out run.

**TODO:**
- [ ] Point `mask_path` at the brain mask.
- [ ] Build `cv = LeaveOneGroupOut()` and the same-style `Decoder`.
- [ ] `decoder.fit(imgs, y, groups=groups)`.
- [ ] Print `decoder.cv_scores_` (per class) and the overall mean accuracy vs 25%.

In [ ]:
from nilearn.decoding import Decoder
from sklearn.model_selection import LeaveOneGroupOut

# func_deriv_dir = project_dir / "Derivatives" / subject / "func"
# TODO: mask_path = ...
# TODO: cv = LeaveOneGroupOut()
# TODO: decoder = Decoder(estimator="svc", mask=mask_path, standardize="zscore_sample",
#                         screening_percentile=..., scoring="accuracy", cv=cv)
# TODO: decoder.fit(imgs, y, groups=groups)
# TODO: print per-class cv_scores_ and the mean vs 25% chance

---
## Section 4 — Get Cross-Validated Predictions for a Confusion Matrix

A confusion matrix compares **true** labels to **predicted** labels. You need a
prediction for every trial, made by a model that did *not* train on that trial —
i.e. **cross-validated** predictions. The clean way to get these is
scikit-learn's `cross_val_predict`, which returns one out-of-fold prediction per
sample.

There is a wrinkle: `cross_val_predict` needs an estimator that consumes a
feature *matrix* `X`, but our images are NIfTI files. So first turn the images
into `X` with a masker, then run a plain scikit-learn pipeline. This also makes
the masking step — usually hidden inside `Decoder` — explicit, which is good for
learning.

```python
from nilearn.maskers import NiftiMasker

masker = NiftiMasker(mask_img=mask_path, standardize="zscore_sample")
X = masker.fit_transform(imgs)     # shape (64, n_voxels) -> the feature matrix
print(X.shape)
```

Then a pipeline that mirrors the `Decoder` internals (ANOVA screening + linear
SVM), evaluated with the same leave-one-run-out CV:

```python
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict

pipe = Pipeline([
    ("anova", SelectPercentile(f_classif, percentile=...)),   # TODO: match your decoder
    ("svc",   SVC(kernel="linear")),
])

y_pred = cross_val_predict(pipe, X, y, groups=groups, cv=LeaveOneGroupOut())
```

Because `cross_val_predict` refits the whole `pipe` (ANOVA included) inside each
fold, the feature selection stays leak-free — same principle as Section 5 of
notebook 01.

**TODO:**
- [ ] Build `masker` and transform `imgs` into `X`; print `X.shape`.
- [ ] Build the `pipe` with `SelectPercentile` + linear `SVC`.
- [ ] Get `y_pred` via `cross_val_predict` with `groups` and `LeaveOneGroupOut`.

In [ ]:
from nilearn.maskers import NiftiMasker
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict

# TODO: masker = NiftiMasker(mask_img=mask_path, standardize="zscore_sample")
# TODO: X = masker.fit_transform(imgs);  print(X.shape)

# TODO: pipe = Pipeline([... ANOVA ..., ... linear SVC ...])
# TODO: y_pred = cross_val_predict(pipe, X, y, groups=groups, cv=LeaveOneGroupOut())

---
## Section 5 — Build and Plot the Confusion Matrix

A **confusion matrix** is a square table: rows = true class, columns = predicted
class. The diagonal counts correct predictions; off-diagonal cells show *which*
class a true class gets mistaken for. Reading the off-diagonal is the whole point.

```python
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["pos_val", "neg_val", "pos_aro", "neg_aro"]   # fix the order explicitly
cm = confusion_matrix(y, y_pred, labels=labels)
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues", colorbar=True)
```

Fixing the `labels` order matters — otherwise rows/columns come out alphabetical
and your reading of "valence block vs arousal block" gets scrambled.

**TODO:**
- [ ] Choose an explicit `labels` order (group the two valence classes together
      and the two arousal classes together — it makes block structure visible).
- [ ] Compute `cm` and plot it with `ConfusionMatrixDisplay`.
- [ ] Save the figure under `decoding/figures/`.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# TODO: labels = [...]   # explicit order; group valence together, arousal together
# TODO: cm = confusion_matrix(y, y_pred, labels=labels)
# TODO: ConfusionMatrixDisplay(...).plot(...)
# TODO: save to decoding/figures/

---
## Section 6 — Interpreting the Mistakes

Now read the matrix like a scientist, not just a scorekeeper. Ask:

- **Is the diagonal above chance?** With 16 trials per class, pure guessing puts
  ~4 on each diagonal cell. Noticeably more than 4 means that class is decodable.
- **Are confusions structured or random?** If `pos_val` is mostly confused with
  `neg_val` (and `pos_aro` with `neg_aro`), the decoder separates the *dimensions*
  but struggles with *sign within* a dimension — consistent with what notebooks
  01–03 hinted at. If confusions are spread evenly, there is no such structure.
- **Does any one class collapse?** A row that is almost entirely predicted as one
  other class can signal imbalance, a bad fold, or a class that simply isn't
  separable here.

**TODO:**
- [ ] In the markdown cell below, describe the diagonal vs off-diagonal pattern.
- [ ] State whether the confusions look like a "valence block vs arousal block"
      structure, and what that would imply.
- [ ] Connect it back to your binary results: is the multiclass story consistent
      with notebooks 01–03?

*Your reading of the confusion matrix:*

> (write here)

---
## Section 7 — Summary of the Whole Series

You have built a complete teaching decoding workflow for `sub-097`:

| notebook | question | classes | chance |
|---|---|---|---|
| 00 | (prepare the input tables) | — | — |
| 01 | positive vs negative **valence** | 2 | 50% |
| 02 | positive vs negative **arousal** | 2 | 50% |
| 03 | **valence** vs **arousal** dimension | 2 | 50% |
| 04 | all four conditions | 4 | 25% |

**The throughline:** the images never changed. Every different "result" came from
**how you labelled and grouped** the same 64 beta maps, and every honest result
depended on **leak-free cross-validation** that respected run boundaries.

**Where a real study goes next:**
- Repeat across **many subjects** and test the *group* accuracy against chance.
- Run a **permutation test** (shuffle `y` many times) to get an *empirical* chance
  distribution instead of assuming exactly 50% / 25%.
- Swap the whole-brain mask for **ROI masks** to localise *where* the information
  lives.
- Inspect **weight maps / confusion structure** to interpret representational
  geometry — carefully, with the Haufe-et-al. caveat in mind.

**TODO:**
- [ ] Write a few sentences summarising what you found across all four decoders
      for `sub-097`, and what you would do next if this were your own study.

*Your closing summary:*

> (write here)